In [ ]:
# ============================================================
# OVERFLOW PREVENTION BY INPUT SCALING
# ============================================================
#
# This notebook demonstrates two scaling conditions derived from
# the theoretical discussion of overflow prevention in fixed-point
# digital filters.
#
# The purpose is deliberately limited:
#
#   1. Time-domain scaling based on the impulse response.
#
#   2. Frequency-domain scaling for a sinusoidal input.
#
#   3. Visualization of the safe range of the scaling factor beta.
#
# More advanced norm-based scaling criteria are intentionally not
# included here.
#
#
# ============================================================
# BASIC OVERFLOW CONDITION
# ============================================================
#
# Assume that all internal fixed-point values must satisfy
#
#                   |y[n]| < 1.
#
# If the input is bounded according to
#
#                   |x[n]| <= x_max,
#
# and the impulse response between the input and a particular
# internal node is h[n], then
#
#                   |y[n]|
#
#           <= x_max sum_m |h[m]|.
#
# Therefore, a sufficient scaling condition is
#
#                   beta x_max sum_m |h[m]| < 1.
#
# Equivalently,
#
#                   beta < beta_max,time
#
# where
#
#                   beta_max,time
#
#           = 1 / [x_max sum_m |h[m]|].
#
#
# ============================================================
# SINUSOIDAL INPUT
# ============================================================
#
# For a sinusoidal input
#
#                   x[n] = x_max cos(omega_0 n),
#
# the output amplitude at the considered node is determined by
#
#                   |H(exp(j omega_0))| x_max.
#
# After applying the scaling factor beta, overflow is avoided if
#
#                   beta x_max |H(exp(j omega_0))| < 1.
#
# Therefore,
#
#                   beta < beta_max,freq
#
# with
#
#                   beta_max,freq
#
#           = 1 / [x_max |H(exp(j omega_0))|].
#
#
# ============================================================
# PRACTICAL POWER-OF-TWO SCALING
# ============================================================
#
# In practical fixed-point systems, beta is frequently selected
# as a negative integer power of two:
#
#                   beta = 2^(-s).
#
# Such scaling can be implemented by a simple binary shift.
#
# The notebook therefore also calculates the largest power-of-two
# scaling coefficient that does not exceed the theoretical bound.
#
#
# ============================================================
# FIR EXAMPLE USED IN THIS NOTEBOOK
# ============================================================
#
# To visualize the theory without introducing a separate filter
# design problem, the notebook uses the illustrative FIR impulse
# response
#
#                   h[n] = a^n,
#
#                   n = 0, 1, ..., M-1.
#
# This is only a pedagogical example.
#
# The theoretical scaling conditions themselves are general and
# are not restricted to this particular FIR sequence.
#
#
# ============================================================
# HOW TO USE THIS NOTEBOOK
# ============================================================
#
# 1. TIME-DOMAIN BOUND
#
#    Active controls:
#
#       x_max
#       FIR parameter a
#       FIR length M
#       beta
#
#    The graph displays the quantity
#
#                   beta x_max sum |h[n]|
#
#    together with the overflow threshold 1.
#
#    Safe operation requires the displayed value to remain below 1.
#
#
# 2. SINUSOIDAL FREQUENCY BOUND
#
#    Active controls:
#
#       x_max
#       FIR parameter a
#       FIR length M
#       omega_0
#       beta
#
#    The graph displays
#
#                   |H(exp(j omega))|
#
#    and marks the selected sinusoidal frequency omega_0.
#
#    The notebook calculates
#
#                   beta x_max |H(exp(j omega_0))|.
#
#    Safe operation requires this quantity to remain below 1.
#
#
# 3. SAFE SCALING REGION
#
#    Active controls:
#
#       x_max
#       FIR parameter a
#       FIR length M
#       omega_0
#       beta
#
#    The graph compares:
#
#       beta_max,time
#
#       beta_max,freq
#
#       selected beta.
#
#    A selected beta lying below the appropriate limit is safe
#    according to the corresponding criterion.
#
#
# ============================================================
# WHAT WE EXPECT TO OBSERVE
# ============================================================
#
# INCREASING x_max
#
# A larger input amplitude leaves less headroom before overflow.
# Therefore, the maximum admissible beta becomes smaller.
#
#
# INCREASING THE INTERNAL GAIN
#
# If the impulse-response magnitude sum or the frequency-response
# magnitude increases, stronger downscaling is required.
#
#
# DECREASING beta
#
# A smaller beta reduces the internal signal level and therefore
# improves overflow protection.
#
# However, excessive downscaling is not desirable because it uses
# less of the available fixed-point dynamic range and can reduce
# the achievable signal-to-noise ratio.
#
#
# POWER-OF-TWO SCALING
#
# The recommended shift-based beta is the largest value
#
#                   beta = 2^(-s)
#
# that remains below the corresponding theoretical bound.
#
#
# ============================================================
# MAIN INTERPRETATION
# ============================================================
#
# Scaling is a compromise.
#
# Too little scaling:
#
#                   risk of overflow.
#
# Too much scaling:
#
#                   unnecessarily small signal levels
#                   and potentially poorer SNR.
#
# The objective is therefore to use the largest safe scaling
# coefficient compatible with the selected overflow criterion.
#
# ============================================================


%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, RadioButtons, VBox, HBox, HTML, Layout, interactive
from IPython.display import display


# ------------------------------------------------------------
# Illustrative FIR impulse response
# ------------------------------------------------------------

def fir_impulse_response(a, M):

    n = np.arange(M)

    h = a**n

    return n, h


# ------------------------------------------------------------
# FIR frequency response
# ------------------------------------------------------------

def fir_frequency_response(a, M, omega):

    n = np.arange(M)

    H = np.sum((a**n)[None, :] * np.exp(-1j * omega[:, None] * n[None, :]), axis=1)

    return H


# ------------------------------------------------------------
# Time-domain scaling bound
# ------------------------------------------------------------

def time_domain_beta_max(x_max, a, M):

    _, h = fir_impulse_response(a, M)

    h_abs_sum = np.sum(np.abs(h))

    beta_max = 1.0 / (x_max * h_abs_sum)

    return beta_max, h_abs_sum


# ------------------------------------------------------------
# Frequency-domain scaling bound
# ------------------------------------------------------------

def frequency_domain_beta_max(x_max, a, M, omega_0):

    n = np.arange(M)

    H_omega_0 = np.sum((a**n) * np.exp(-1j * omega_0 * n))

    magnitude = np.abs(H_omega_0)

    beta_max = 1.0 / (x_max * magnitude)

    return beta_max, magnitude


# ------------------------------------------------------------
# Largest safe power-of-two scaling coefficient
# ------------------------------------------------------------

def safe_power_of_two(beta_max):

    if beta_max >= 1.0:

        return 1.0, 0

    shift = int(np.ceil(-np.log2(beta_max)))

    beta_shift = 2.0**(-shift)

    return beta_shift, shift


# ------------------------------------------------------------
# Style
# ------------------------------------------------------------

style_html = HTML("""
<style>

.os-root {
    font-family: monospace;
    width: 960px;
    max-width: 960px;
}

.os-description {
    font-size: 13px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 8px;
    box-sizing: border-box;
}

.os-howto {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #c7cfe0;
    border-left: 6px solid #667da8;
    background: #f8f9fc;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.os-howto-title {
    font-size: 13px;
    font-weight: bold;
    color: #40587d;
    margin-bottom: 5px;
}

.os-observe {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #c7d8c9;
    border-left: 6px solid #3c8a4e;
    background: #f7fbf7;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.os-observe-title {
    font-size: 13px;
    font-weight: bold;
    color: #245c31;
    margin-bottom: 5px;
}

.os-model {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 9px 12px;
    border: 1px solid #d5c58a;
    border-left: 6px solid #b8860b;
    background: #fffaf0;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.os-model-title {
    font-size: 13px;
    font-weight: bold;
    color: #8a6500;
    margin-bottom: 5px;
}

.os-box {
    border: 1px solid #c8d0dc;
    border-radius: 9px;
    padding: 9px 12px;
    box-sizing: border-box;
}

.os-title {
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.os-info {
    font-size: 13px;
    line-height: 1.52;
}

.os-label {
    display: inline-block;
    min-width: 245px;
    font-weight: bold;
}

.os-value {
    font-size: 14px;
    font-weight: bold;
}

.os-safe {
    color: #176b34;
    font-weight: bold;
}

.os-overflow {
    color: #a32828;
    font-weight: bold;
}

.os-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    margin-top: 6px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

title_html = HTML("""
<div class="os-root">

    <div style="
        font-family:monospace;
        font-size:22px;
        font-weight:bold;
        margin-bottom:8px;
    ">
        Overflow Prevention by Input Scaling
    </div>

</div>
""")


# ------------------------------------------------------------
# Visible introductory documentation
# ------------------------------------------------------------

description_html = HTML("""
<div class="os-root">

    <div class="os-description">

        <b>What this notebook demonstrates</b><br><br>

        For a bounded input

        <div style="text-align:center; margin:6px 0;">
            <b>|x[n]| ≤ x<sub>max</sub>,</b>
        </div>

        the time-domain overflow bound at an internal node is

        <div style="text-align:center; margin:6px 0;">
            <b>
            β x<sub>max</sub> Σ |h[n]| &lt; 1.
            </b>
        </div>

        For a sinusoidal input with frequency ω<sub>0</sub>, the corresponding
        frequency-domain condition is

        <div style="text-align:center; margin:6px 0;">
            <b>
            β x<sub>max</sub> |H(e<sup>jω<sub>0</sub></sup>)| &lt; 1.
            </b>
        </div>

        The notebook visualizes these two conditions and the associated
        maximum admissible scaling coefficient β.

    </div>


    <div class="os-howto">

        <div class="os-howto-title">
            How to use this notebook
        </div>

        <b>Time-domain bound:</b>
        vary x<sub>max</sub>, the FIR parameters and β. The displayed
        quantity βx<sub>max</sub>Σ|h[n]| must remain below the overflow
        threshold 1.<br><br>

        <b>Sinusoidal frequency bound:</b>
        select the sinusoidal frequency ω<sub>0</sub> and inspect the FIR
        magnitude response. The selected operating frequency is marked on
        the curve and the condition
        βx<sub>max</sub>|H(e<sup>jω<sub>0</sub></sup>)| &lt; 1 is evaluated
        numerically.<br><br>

        <b>Safe scaling region:</b>
        compare the selected β with the maximum values permitted by the
        time-domain and sinusoidal criteria.<br><br>

        Controls that do not affect the selected display are automatically
        disabled.

    </div>


    <div class="os-observe">

        <div class="os-observe-title">
            What we expect to observe
        </div>

        Increasing <b>x<sub>max</sub></b> reduces the permissible scaling
        coefficient because less internal headroom remains before
        overflow.<br><br>

        Increasing the gain between the input and the considered internal
        node also reduces the maximum permissible β.<br><br>

        Decreasing β improves overflow protection. Excessive downscaling,
        however, unnecessarily reduces the signal level and may degrade
        the signal-to-noise ratio.<br><br>

        The practical objective is therefore to use the <b>largest safe
        scaling coefficient</b> compatible with the selected criterion.

    </div>


    <div class="os-model">

        <div class="os-model-title">
            Illustrative FIR used in this demonstration
        </div>

        To visualize the scaling conditions without introducing a separate
        filter-design problem, the notebook uses

        <div style="text-align:center; margin:6px 0;">
            <b>
            h[n] = a<sup>n</sup>,
            &nbsp;&nbsp;
            n = 0,1,...,M−1.
            </b>
        </div>

        This FIR sequence is used only as an interactive example. The
        overflow bounds demonstrated above are the general relations from
        the theoretical development.<br><br>

        A practical shift-based coefficient is also reported in the form

        <div style="text-align:center; margin:6px 0;">
            <b>β = 2<sup>−s</sup>,</b>
        </div>

        chosen as the largest power-of-two value that does not exceed the
        theoretical scaling limit.

    </div>

</div>
""")


# ------------------------------------------------------------
# Dynamic summary
# ------------------------------------------------------------

summary_html = HTML()

summary_html.layout = Layout(
    width='610px',
    min_width='610px',
    overflow='visible'
)


# ------------------------------------------------------------
# Main interactive function
# ------------------------------------------------------------

def plot_scaling(display_mode='Time-domain bound', x_max=0.80, a=0.75, M=8, omega_0=0.40 * np.pi, beta=0.25):


    # --------------------------------------------------------
    # Theoretical bounds
    # --------------------------------------------------------

    beta_time, h_abs_sum = time_domain_beta_max(x_max, a, M)

    beta_freq, H_selected = frequency_domain_beta_max(x_max, a, M, omega_0)


    # --------------------------------------------------------
    # Practical power-of-two scaling values
    # --------------------------------------------------------

    beta_time_shift, shift_time = safe_power_of_two(beta_time)

    beta_freq_shift, shift_freq = safe_power_of_two(beta_freq)


    # --------------------------------------------------------
    # Current normalized internal amplitudes
    # --------------------------------------------------------

    time_quantity = beta * x_max * h_abs_sum

    frequency_quantity = beta * x_max * H_selected


    # --------------------------------------------------------
    # Safety assessment
    # --------------------------------------------------------

    if time_quantity < 1.0:

        time_status = "<span class='os-safe'>SAFE</span>"

    else:

        time_status = "<span class='os-overflow'>OVERFLOW POSSIBLE</span>"


    if frequency_quantity < 1.0:

        frequency_status = "<span class='os-safe'>SAFE</span>"

    else:

        frequency_status = "<span class='os-overflow'>OVERFLOW POSSIBLE</span>"


    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    summary_html.value = f"""
    <div class="os-box">

        <div class="os-title">
            Current Scaling Data
        </div>

        <div class="os-info">

            <span class="os-label">Selected display</span>
            {display_mode}
            <br>

            <span class="os-label">Maximum input magnitude</span>
            x<sub>max</sub> = <span class="os-value">{x_max:.4f}</span>
            <br>

            <span class="os-label">FIR parameter</span>
            a = {a:.4f}
            <br>

            <span class="os-label">FIR length</span>
            M = {M}
            <br>

            <span class="os-label">Selected scaling coefficient</span>
            β = <span class="os-value">{beta:.6f}</span>
            <br><br>

            <b>Time-domain criterion</b>
            <br>

            <span class="os-label">Σ |h[n]|</span>
            {h_abs_sum:.8f}
            <br>

            <span class="os-label">Maximum theoretical β</span>
            {beta_time:.8f}
            <br>

            <span class="os-label">Largest safe 2<sup>−s</sup></span>
            β = {beta_time_shift:.8f}
            &nbsp;&nbsp;(s = {shift_time})
            <br>

            <span class="os-label">β x<sub>max</sub> Σ|h[n]|</span>
            {time_quantity:.8f}
            <br>

            <span class="os-label">Time-domain status</span>
            {time_status}
            <br><br>

            <b>Sinusoidal criterion</b>
            <br>

            <span class="os-label">Selected frequency</span>
            ω<sub>0</sub> = {omega_0 / np.pi:.3f}π
            <br>

            <span class="os-label">|H(e<sup>jω0</sup>)|</span>
            {H_selected:.8f}
            <br>

            <span class="os-label">Maximum theoretical β</span>
            {beta_freq:.8f}
            <br>

            <span class="os-label">Largest safe 2<sup>−s</sup></span>
            β = {beta_freq_shift:.8f}
            &nbsp;&nbsp;(s = {shift_freq})
            <br>

            <span class="os-label">β x<sub>max</sub>|H(e<sup>jω0</sup>)|</span>
            {frequency_quantity:.8f}
            <br>

            <span class="os-label">Frequency-domain status</span>
            {frequency_status}

        </div>

        <div class="os-note">
            Strictly speaking, the theoretical conditions use a strict
            inequality. The displayed power-of-two coefficient is therefore
            interpreted as a practical conservative scaling choice.
        </div>

    </div>
    """


    # --------------------------------------------------------
    # One large figure
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(11.8, 5.0)
    )


    # ========================================================
    # TIME-DOMAIN BOUND
    # ========================================================

    if display_mode == 'Time-domain bound':

        beta_values = np.linspace(
            0.0,
            max(1.05, 1.20 * beta_time),
            800
        )


        internal_bound = beta_values * x_max * h_abs_sum


        ax.plot(
            beta_values,
            internal_bound,
            linewidth=2.0,
            label=r'$\beta x_{\max}\sum |h[n]|$'
        )


        ax.axhline(
            1.0,
            linestyle='--',
            linewidth=1.5,
            label='Overflow threshold'
        )


        ax.axvline(
            beta_time,
            linestyle=':',
            linewidth=1.5,
            label=r'$\beta_{\max,\mathrm{time}}$'
        )


        ax.plot(
            beta,
            time_quantity,
            'o',
            markersize=9,
            label='Selected β'
        )


        ax.set_xlabel(
            'Scaling coefficient β'
        )


        ax.set_ylabel(
            r'$\beta x_{\max}\sum |h[n]|$'
        )


        ax.set_title(
            'Time-Domain Overflow Bound',
            fontsize=12
        )


        ax.grid(
            True,
            linestyle=':',
            alpha=0.5
        )


        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            ncol=4,
            frameon=False,
            fontsize=9
        )


    # ========================================================
    # SINUSOIDAL FREQUENCY BOUND
    # ========================================================

    elif display_mode == 'Sinusoidal frequency bound':

        omega = np.linspace(
            0.0,
            np.pi,
            1200
        )


        H = fir_frequency_response(
            a,
            M,
            omega
        )


        magnitude = np.abs(
            H
        )


        ax.plot(
            omega / np.pi,
            magnitude,
            linewidth=2.0,
            label=r'$|H(e^{j\omega})|$'
        )


        ax.plot(
            omega_0 / np.pi,
            H_selected,
            'o',
            markersize=9,
            label='Selected frequency'
        )


        ax.axvline(
            omega_0 / np.pi,
            linestyle='--',
            linewidth=1.0
        )


        ax.set_xlim(
            0.0,
            1.0
        )


        ax.set_xlabel(
            'Normalized frequency ω / π'
        )


        ax.set_ylabel(
            r'$|H(e^{j\omega})|$'
        )


        ax.set_title(
            'Magnitude Response Used for Sinusoidal Scaling',
            fontsize=12
        )


        ax.grid(
            True,
            linestyle=':',
            alpha=0.5
        )


        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            ncol=2,
            frameon=False,
            fontsize=9
        )


    # ========================================================
    # SAFE SCALING REGION
    # ========================================================

    else:

        positions = np.arange(
            2
        )


        limits = [
            beta_time,
            beta_freq
        ]


        labels = [
            'Time-domain\nlimit',
            'Sinusoidal\nfrequency limit'
        ]


        ax.bar(
            positions,
            limits,
            width=0.52,
            alpha=0.75,
            label='Maximum theoretical β'
        )


        ax.axhline(
            beta,
            linestyle='--',
            linewidth=1.8,
            label='Selected β'
        )


        ax.set_xticks(
            positions
        )


        ax.set_xticklabels(
            labels
        )


        ax.set_ylabel(
            'Scaling coefficient β'
        )


        ax.set_title(
            'Safe Scaling Limits',
            fontsize=12
        )


        maximum_value = max(
            beta_time,
            beta_freq,
            beta
        )


        ax.set_ylim(
            0.0,
            1.20 * maximum_value
        )


        ax.grid(
            True,
            axis='y',
            linestyle=':',
            alpha=0.5
        )


        for position, value in zip(positions, limits):

            ax.text(
                position,
                value + 0.025 * maximum_value,
                f'{value:.4f}',
                ha='center',
                va='bottom',
                fontsize=10
            )


        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            ncol=2,
            frameon=False,
            fontsize=9
        )


    # --------------------------------------------------------
    # Final spacing
    # --------------------------------------------------------

    plt.subplots_adjust(
        left=0.09,
        right=0.98,
        top=0.90,
        bottom=0.24
    )


    plt.show()

    plt.close(fig)


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(
    width='295px'
)


slider_style = {
    'description_width': '115px'
}


display_selector = RadioButtons(
    options=[
        'Time-domain bound',
        'Sinusoidal frequency bound',
        'Safe scaling region'
    ],
    value='Time-domain bound',
    description='Display:',
    style={'description_width': '65px'},
    layout=Layout(
        width='310px'
    )
)


x_max_slider = FloatSlider(
    value=0.80,
    min=0.10,
    max=1.00,
    step=0.01,
    description='Input x_max:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2f'
)


a_slider = FloatSlider(
    value=0.75,
    min=0.00,
    max=0.95,
    step=0.01,
    description='FIR parameter a:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2f'
)


M_slider = IntSlider(
    value=8,
    min=2,
    max=32,
    step=1,
    description='FIR length M:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


omega_slider = FloatSlider(
    value=0.40 * np.pi,
    min=0.0,
    max=np.pi,
    step=np.pi / 100.0,
    description='Frequency ω0:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.3f'
)


beta_slider = FloatSlider(
    value=0.25,
    min=0.01,
    max=1.00,
    step=0.01,
    description='Scaling beta:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2f'
)


# ------------------------------------------------------------
# Enable only relevant controls
# ------------------------------------------------------------

def update_control_states(change=None):

    mode = display_selector.value


    if mode == 'Time-domain bound':

        x_max_slider.disabled = False

        a_slider.disabled = False

        M_slider.disabled = False

        omega_slider.disabled = True

        beta_slider.disabled = False


    elif mode == 'Sinusoidal frequency bound':

        x_max_slider.disabled = False

        a_slider.disabled = False

        M_slider.disabled = False

        omega_slider.disabled = False

        beta_slider.disabled = False


    else:

        x_max_slider.disabled = False

        a_slider.disabled = False

        M_slider.disabled = False

        omega_slider.disabled = False

        beta_slider.disabled = False


display_selector.observe(
    update_control_states,
    names='value'
)


# ------------------------------------------------------------
# Initial state
# ------------------------------------------------------------

update_control_states()


# ------------------------------------------------------------
# Interactive object
# ------------------------------------------------------------

widget_plot = interactive(
    plot_scaling,
    display_mode=display_selector,
    x_max=x_max_slider,
    a=a_slider,
    M=M_slider,
    omega_0=omega_slider,
    beta=beta_slider
)


# ------------------------------------------------------------
# Controls box
# ------------------------------------------------------------

controls_box = VBox(
    [
        HTML("<div class='os-title'>Controls</div>"),
        display_selector,
        x_max_slider,
        a_slider,
        M_slider,
        omega_slider,
        beta_slider
    ],
    layout=Layout(
        width='335px',
        min_width='335px',
        border='1px solid #c8d0dc',
        padding='9px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Summary + controls
# ------------------------------------------------------------

top_row = HBox(
    [
        summary_html,
        controls_box
    ],
    layout=Layout(
        width='960px',
        max_width='960px',
        overflow='visible',
        align_items='flex-start',
        justify_content='space-between'
    )
)


# ------------------------------------------------------------
# Plot output
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

plot_output.layout = Layout(
    width='auto',
    overflow='visible'
)


# ------------------------------------------------------------
# Final notebook layout
# ------------------------------------------------------------

main_layout = VBox(
    [
        description_html,
        top_row,
        plot_output
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(title_html)

display(main_layout)